# Notebook 05 — Results Summary & Key Findings
## Machine Learning for Product Gain/Loss Prediction
**Author:** Uzoma Nnaemeka Eze | MSc Data Science | UE Potsdam

---
This notebook consolidates all model results and presents the key findings of the study.

## Summary of Results

### Depot-Level Models (M2 Daily Summary — 365 records per product)

| Model | Product | MAE (L) | RMSE (L) | R² / AUC |
|---|---|---|---|---|
| Linear Regression | AGO | 9,655 | 11,712 | R² = -0.3193 |
| Decision Tree Regressor | AGO | 7,630 | 11,338 | R² = -0.2363 |
| Logistic Regression | AGO | — | — | AUC = 0.6842 |
| Decision Tree Classifier | AGO | — | — | AUC = 0.3816 |
| Linear Regression | PMS | 10,294 | 15,023 | R² = -0.0241 |
| Decision Tree Regressor | PMS | 13,001 | 20,779 | R² = -0.9590 |
| Logistic Regression | PMS | — | — | AUC = 0.6852 |
| Decision Tree Classifier | PMS | — | — | AUC = 0.7013 |

### Tank-Level Models (D2 Daily Tank Reports — 4,264 tank-day records)

| Model | Product | AUC | Recall (Loss) | TP / FN |
|---|---|---|---|---|
| Logistic Regression | AGO | **0.8312** | **95.0%** | 19 / 1 |
| Decision Tree Classifier | AGO | 0.7160 | 75.0% | 15 / 5 |
| Logistic Regression | PMS | 0.6970 | 79.2% | 19 / 5 |
| Decision Tree Classifier | PMS | 0.5040 | 45.8% | 11 / 13 |


## Key Finding 1 — Regression Limitations

In [25]:
print("""
KEY FINDING 1 — REGRESSION TASK
================================
All regression models returned negative R² values at both M2 and D2 levels.

This is NOT a failure of the models or the dataset.

It reflects a fundamental characteristic of dip-based gauging systems:
- The Loss/Gain figure is the residual of two large imprecise measurements
- Day-to-day variability is dominated by measurement system variance
- No operational summary variable can absorb the noise this introduces

Implication: Continuous volumetric prediction from routine depot records
alone is not technically feasible without higher-frequency instrumentation.
""")



KEY FINDING 1 — REGRESSION TASK
All regression models returned negative R² values at both M2 and D2 levels.

This is NOT a failure of the models or the dataset.

It reflects a fundamental characteristic of dip-based gauging systems:
- The Loss/Gain figure is the residual of two large imprecise measurements
- Day-to-day variability is dominated by measurement system variance
- No operational summary variable can absorb the noise this introduces

Implication: Continuous volumetric prediction from routine depot records
alone is not technically feasible without higher-frequency instrumentation.



## Key Finding 2 — Classification Performance

In [26]:
print("""
KEY FINDING 2 — CLASSIFICATION TASK
=====================================
Binary classification (Loss vs No-Loss) is substantially more tractable.

HEADLINE RESULT:
AGO Tank-Level Logistic Regression
  AUC    = 0.8312
  Recall = 95.0%  (19 out of 20 actual loss events correctly identified)

This result demonstrates that machine learning trained on daily tank-level
operational records can provide meaningful early warning of loss events.

The improvement from depot-level (AUC 0.6842) to tank-level (AUC 0.8312)
for AGO confirms that individual tank dip readings in millimetres add
substantial predictive value over summary-level features.
""")



KEY FINDING 2 — CLASSIFICATION TASK
Binary classification (Loss vs No-Loss) is substantially more tractable.

HEADLINE RESULT:
AGO Tank-Level Logistic Regression
  AUC    = 0.8312
  Recall = 95.0%  (19 out of 20 actual loss events correctly identified)

This result demonstrates that machine learning trained on daily tank-level
operational records can provide meaningful early warning of loss events.

The improvement from depot-level (AUC 0.6842) to tank-level (AUC 0.8312)
for AGO confirms that individual tank dip readings in millimetres add
substantial predictive value over summary-level features.



## Key Finding 3 — Dip Threshold Operational Decision Table

In [27]:
import pandas as pd

dip_table = pd.DataFrame({
    'Dip Range (mm)':    ['0 to 1,000', '1,001 to 2,000', '2,001 to 3,000',
                          '3,001 to 5,000', '5,001 to 7,000', '7,001 to 9,000', '9,001+'],
    'AGO Count':         [260, 359, 98, 87, 62, 79, 126],
    'AGO Loss Prob (%)': [1.9, 6.4, 13.3, 23.0, 14.5, 19.0, 20.6],
    'AGO Risk':          ['LOW','LOW','MODERATE','HIGH','MODERATE','HIGH','HIGH'],
    'PMS Count':         ['N/A', 216, 94, 244, 112, 96, 156],
    'PMS Loss Prob (%)': ['N/A', 1.9, 18.1, 10.2, 21.4, 15.6, 28.2],
    'PMS Risk':          ['N/A','LOW','ELEVATED','MODERATE','HIGH','MODERATE','HIGHEST'],
})

print("DIP THRESHOLD OPERATIONAL DECISION TABLE")
print("="*80)
print(dip_table.to_string(index=False))
print()
print("OPERATIONAL RECOMMENDATION:")
print("  AGO: Increase inspection frequency when dip is 3,001 to 5,000 mm (23% loss risk)")
print("  PMS: Increase inspection frequency when dip is above 9,000 mm (28.2% loss risk)")


DIP THRESHOLD OPERATIONAL DECISION TABLE
Dip Range (mm)  AGO Count  AGO Loss Prob (%) AGO Risk PMS Count PMS Loss Prob (%) PMS Risk
    0 to 1,000        260                1.9      LOW       N/A               N/A      N/A
1,001 to 2,000        359                6.4      LOW       216               1.9      LOW
2,001 to 3,000         98               13.3 MODERATE        94              18.1 ELEVATED
3,001 to 5,000         87               23.0     HIGH       244              10.2 MODERATE
5,001 to 7,000         62               14.5 MODERATE       112              21.4     HIGH
7,001 to 9,000         79               19.0     HIGH        96              15.6 MODERATE
        9,001+        126               20.6     HIGH       156              28.2  HIGHEST

OPERATIONAL RECOMMENDATION:
  AGO: Increase inspection frequency when dip is 3,001 to 5,000 mm (23% loss risk)
  PMS: Increase inspection frequency when dip is above 9,000 mm (28.2% loss risk)


## Key Finding 4 — Pareto Analysis

In [28]:
print("""
KEY FINDING 4 — PARETO ANALYSIS
=================================
AGO: 29 loss events (38.7% of 75 total) account for 80% of annual loss volume
     Total annual AGO loss: 358,659 litres
     Largest single event: 37,759 litres (8 January 2025)

PMS: 49 loss events (43.8% of 112 total) account for 80% of annual loss volume
     Total annual PMS loss: 672,330 litres
     Largest single event: 41,939 litres (31 October 2025)

IMPLICATION:
A targeted detection strategy focusing on the top 40% of loss events
by magnitude would recover the vast majority of annual loss value.
The AGO tank-level Logistic Regression (recall 95%) is best deployed
as an early-warning tool for exactly these high-magnitude events.
""")



KEY FINDING 4 — PARETO ANALYSIS
AGO: 29 loss events (38.7% of 75 total) account for 80% of annual loss volume
     Total annual AGO loss: 358,659 litres
     Largest single event: 37,759 litres (8 January 2025)

PMS: 49 loss events (43.8% of 112 total) account for 80% of annual loss volume
     Total annual PMS loss: 672,330 litres
     Largest single event: 41,939 litres (31 October 2025)

IMPLICATION:
A targeted detection strategy focusing on the top 40% of loss events
by magnitude would recover the vast majority of annual loss value.
The AGO tank-level Logistic Regression (recall 95%) is best deployed
as an early-warning tool for exactly these high-magnitude events.

